# Deep Learning vs Shallow ML — Oil Recovery Factor Prediction
## MLP · CNN-1D · LSTM · Transformer  vs  RF · XGBoost · SVR · GradientBoosting

**Dataset:** Proxy5 — polymer flood reservoir simulation  
**Framework:** Keras / TensorFlow (DL) + scikit-learn / XGBoost (shallow ML)  

All 8 models are evaluated on the same 15% held-out test set using RMSE, MAE, R², and MAPE.

---
## 1. Imports & Configuration

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

import tensorflow as tf
import keras
from keras import layers, Model, Input
from keras.callbacks import EarlyStopping, ReduceLROnPlateau

from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

try:
    import xgboost as xgb
    XGB_AVAILABLE = True
    print(f'XGBoost : {xgb.__version__}')
except ImportError:
    print('XGBoost not installed — run: pip install xgboost')
    XGB_AVAILABLE = False

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

print(f'TensorFlow : {tf.__version__}')
print(f'Keras      : {keras.__version__}')
print(f'GPU        : {bool(tf.config.list_physical_devices("GPU"))}')

BATCH_SIZE = 64
EPOCHS     = 200
LR         = 1e-3
PATIENCE   = 20
TEST_SIZE  = 0.15
VAL_SIZE   = 0.15

# Colour palette — DL models (cool), ML models (warm)
COLOR_MAP = {
    'MLP':              '#2196F3',
    'CNN-1D':           '#FF9800',
    'LSTM':             '#4CAF50',
    'Transformer':      '#9C27B0',
    'Random Forest':    '#F44336',
    'XGBoost':          '#795548',
    'SVR':              '#00BCD4',
    'Gradient Boosting':'#FF5722',
}

DL_MODELS = ['MLP', 'CNN-1D', 'LSTM', 'Transformer']
ML_MODELS = ['Random Forest', 'XGBoost', 'SVR', 'Gradient Boosting']
ALL_MODELS = DL_MODELS + ML_MODELS


def draw_architecture(blocks, title, color, filename):
    """Pure-matplotlib block diagram — no pydot/graphviz needed."""
    n = len(blocks)
    fig, ax = plt.subplots(figsize=(6, max(5, n * 0.85 + 1.2)))
    ax.set_xlim(0, 10)
    ax.set_ylim(0, n + 1)
    ax.axis('off')
    box_w, box_h, x0 = 7.0, 0.60, 1.5
    for i, (label, shape) in enumerate(blocks):
        y = n - i
        ax.add_patch(mpatches.FancyBboxPatch(
            (x0, y - box_h/2), box_w, box_h,
            boxstyle='round,pad=0.05',
            facecolor=color, edgecolor='white', alpha=0.85, linewidth=1.5))
        ax.text(5, y,        label, ha='center', va='center',
                fontsize=9, fontweight='bold', color='white')
        ax.text(5, y - 0.26, shape, ha='center', va='center',
                fontsize=7, color='white', alpha=0.9)
        if i < n - 1:
            y_next = n - (i + 1)
            ax.annotate('', xy=(5, y_next + box_h/2 + 0.04),
                        xytext=(5, y - box_h/2 - 0.04),
                        arrowprops=dict(arrowstyle='->', color='#555', lw=1.5))
    ax.set_title(title, fontsize=12, fontweight='bold', pad=8)
    plt.tight_layout()
    plt.savefig(filename, dpi=150, bbox_inches='tight')
    plt.show()

---
## 2. Data Loading & EDA

In [ ]:
df = pd.read_csv('Proxy5.csv', encoding='latin_1').dropna()
print(f'Shape: {df.shape}')

TARGET   = 'Oil_recovery_factor (%)'
FEATURES = [c for c in df.columns if c != TARGET]
print(f'Features ({len(FEATURES)}):', FEATURES)
df.head()

In [ ]:
df.describe().T.style.background_gradient(cmap='YlGnBu', axis=1)

In [ ]:
# ── Fig 1: Target distribution ───────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
axes[0].hist(df[TARGET], bins=60, color='steelblue', edgecolor='white')
axes[0].set_title('Histogram', fontsize=12)
axes[0].set_xlabel(TARGET)

axes[1].boxplot(df[TARGET], vert=True, patch_artist=True,
                boxprops=dict(facecolor='lightblue'),
                medianprops=dict(color='navy', linewidth=2))
axes[1].set_title('Box Plot', fontsize=12)
axes[1].set_xticklabels(['Oil Recovery Factor'])

sv = np.sort(df[TARGET])
axes[2].plot(sv, np.arange(1, len(sv)+1)/len(sv), lw=2, color='steelblue')
axes[2].set_title('CDF', fontsize=12)
axes[2].set_xlabel(TARGET)
axes[2].grid(alpha=0.3)

plt.suptitle('Target Variable Distribution', fontweight='bold', fontsize=14)
plt.tight_layout()
plt.savefig('fig01_target_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Fig 2: Feature distributions ─────────────────────────────────────────────
fig, axes = plt.subplots(3, 5, figsize=(20, 12))
axes = axes.ravel()
for i, f in enumerate(FEATURES):
    axes[i].hist(df[f], bins=40, color='#5C85D6', edgecolor='white', alpha=0.85)
    axes[i].set_title(f, fontweight='bold', fontsize=8)
    axes[i].tick_params(labelsize=7)
for j in range(len(FEATURES), len(axes)):
    axes[j].set_visible(False)
plt.suptitle('Feature Distributions', fontweight='bold', fontsize=14)
plt.tight_layout()
plt.savefig('fig02_feature_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Fig 3: Correlation heatmap ────────────────────────────────────────────────
plt.figure(figsize=(15, 11))
corr = df.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            vmin=-1, vmax=1, square=True, linewidths=0.4, annot_kws={'size': 7})
plt.title('Feature Correlation Matrix', fontweight='bold', fontsize=14)
plt.tight_layout()
plt.savefig('fig03_correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Fig 4: Features vs Target ─────────────────────────────────────────────────
fig, axes = plt.subplots(3, 5, figsize=(20, 12))
axes = axes.ravel()
for i, f in enumerate(FEATURES):
    axes[i].scatter(df[f], df[TARGET], s=6, alpha=0.15, color='#E05C5C')
    axes[i].set_xlabel(f, fontsize=7)
    axes[i].set_ylabel('Recovery (%)', fontsize=7)
    axes[i].tick_params(labelsize=6)
    r = np.corrcoef(df[f], df[TARGET])[0, 1]
    axes[i].set_title(f'r = {r:.3f}', fontweight='bold', fontsize=8)
for j in range(len(FEATURES), len(axes)):
    axes[j].set_visible(False)
plt.suptitle('Features vs Oil Recovery Factor', fontweight='bold', fontsize=14)
plt.tight_layout()
plt.savefig('fig04_feature_vs_target.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 3. Pre-processing

In [ ]:
X = df[FEATURES].values.astype(np.float32)
y = df[TARGET].values.astype(np.float32).reshape(-1, 1)

X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=SEED)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=VAL_SIZE/(1-TEST_SIZE), random_state=SEED)

print(f'Train: {X_train.shape[0]} | Val: {X_val.shape[0]} | Test: {X_test.shape[0]}')

x_scaler = StandardScaler()
y_scaler = StandardScaler()

X_train_s = x_scaler.fit_transform(X_train)
X_val_s   = x_scaler.transform(X_val)
X_test_s  = x_scaler.transform(X_test)

y_train_s = y_scaler.fit_transform(y_train)
y_val_s   = y_scaler.transform(y_val)
y_test_s  = y_scaler.transform(y_test)

N_FEATURES = X_train_s.shape[1]
print(f'Input dimension: {N_FEATURES}')

# Sequence shape for DL models: (batch, 1, N_FEATURES)
X_train_seq = X_train_s.reshape(-1, 1, N_FEATURES)
X_val_seq   = X_val_s.reshape(-1, 1, N_FEATURES)
X_test_seq  = X_test_s.reshape(-1, 1, N_FEATURES)

# Flat 1-D targets for sklearn (no scaler needed — sklearn models use raw targets)
y_train_flat = y_train.ravel()
y_val_flat   = y_val.ravel()
y_test_flat  = y_test.ravel()

In [ ]:
# ── Fig 5: Dataset split pie ──────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 4))
sizes  = [X_train.shape[0], X_val.shape[0], X_test.shape[0]]
labels = ['Train (70%)', 'Validation (15%)', 'Test (15%)']
colors = ['#4CAF50', '#FF9800', '#F44336']
wedges, texts, autotexts = ax.pie(
    sizes, labels=labels, colors=colors,
    autopct='%1.1f%%', startangle=140,
    wedgeprops=dict(edgecolor='white', linewidth=2))
for at in autotexts:
    at.set_fontsize(11)
ax.set_title(f'Dataset Split  (n = {len(df)})', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('fig05_dataset_split.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 4. DL Training Utilities

In [ ]:
def get_callbacks():
    return [
        EarlyStopping(monitor='val_loss', patience=PATIENCE,
                      restore_best_weights=True, verbose=1),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                          patience=10, min_lr=1e-6, verbose=0),
    ]


def fit_dl(model, name, X_tr, y_tr, X_va, y_va):
    model.compile(optimizer=keras.optimizers.Adam(LR),
                  loss='mse', metrics=['mae'])
    history = model.fit(
        X_tr, y_tr,
        validation_data=(X_va, y_va),
        epochs=EPOCHS, batch_size=BATCH_SIZE,
        callbacks=get_callbacks(), verbose=0)
    ep  = len(history.history['loss'])
    trL = history.history['loss'][-1]
    vaL = history.history['val_loss'][-1]
    print(f'[{name}] stopped at epoch {ep} — train={trL:.5f}  val={vaL:.5f}')
    return history


def evaluate_dl(model, X_te, y_te_s, name):
    """Evaluate a Keras model; y_te_s is scaled, returns original-scale metrics."""
    pred_s   = model.predict(X_te, verbose=0)
    pred_inv = y_scaler.inverse_transform(pred_s)
    true_inv = y_scaler.inverse_transform(y_te_s)
    return _metrics(true_inv, pred_inv, name)


def evaluate_ml(model, X_te, y_te, name):
    """Evaluate a sklearn model; X_te is scaled, y_te is original-scale."""
    pred = model.predict(X_te).reshape(-1, 1)
    true = y_te.reshape(-1, 1)
    return _metrics(true, pred, name)


def _metrics(true_inv, pred_inv, name):
    rmse = np.sqrt(mean_squared_error(true_inv, pred_inv))
    mae  = mean_absolute_error(true_inv, pred_inv)
    r2   = r2_score(true_inv, pred_inv)
    mape = np.mean(np.abs((true_inv - pred_inv) /
                          np.where(true_inv == 0, 1e-8, true_inv))) * 100
    print(f'[{name}] RMSE={rmse:.4f}  MAE={mae:.4f}  R²={r2:.4f}  MAPE={mape:.2f}%')
    return pred_inv.ravel(), true_inv.ravel(), {'RMSE': rmse, 'MAE': mae, 'R2': r2, 'MAPE': mape}

---
## 5. DL Model 1 — MLP

In [ ]:
def build_mlp(n_features, dropout=0.2):
    inp = Input(shape=(n_features,), name='input')
    x = layers.Dense(128, activation='relu')(inp)
    x = layers.Dropout(dropout)(x)
    x = layers.Dense(64, activation='relu')(x)
    x = layers.Dropout(dropout)(x)
    x = layers.Dense(32, activation='relu')(x)
    out = layers.Dense(1)(x)
    return Model(inp, out, name='MLP')

mlp_model = build_mlp(N_FEATURES)
mlp_model.summary()

In [ ]:
draw_architecture([
    ('Input',                          f'(None, {N_FEATURES})'),
    ('Dense 128 + ReLU + Dropout(0.2)','(None, 128)'),
    ('Dense  64 + ReLU + Dropout(0.2)','(None,  64)'),
    ('Dense  32 + ReLU',               '(None,  32)'),
    ('Dense   1  (Output)',            '(None,   1)'),
], title='MLP Architecture', color=COLOR_MAP['MLP'],
   filename='fig06a_mlp_arch.png')

In [ ]:
mlp_history = fit_dl(mlp_model, 'MLP', X_train_s, y_train_s, X_val_s, y_val_s)
mlp_preds, mlp_trues, mlp_metrics = evaluate_dl(mlp_model, X_test_s, y_test_s, 'MLP')

---
## 6. DL Model 2 — CNN-1D

In [ ]:
def build_cnn(n_features, dropout=0.2):
    inp = Input(shape=(1, n_features), name='input')
    x = layers.ZeroPadding1D(padding=1)(inp)
    x = layers.Conv1D(64, kernel_size=3, activation='relu')(x)
    x = layers.Dropout(dropout)(x)
    x = layers.ZeroPadding1D(padding=1)(x)
    x = layers.Conv1D(32, kernel_size=3, activation='relu')(x)
    x = layers.Flatten()(x)
    x = layers.Dense(32, activation='relu')(x)
    x = layers.Dropout(dropout)(x)
    out = layers.Dense(1)(x)
    return Model(inp, out, name='CNN_1D')

cnn_model = build_cnn(N_FEATURES)
cnn_model.summary()

In [ ]:
draw_architecture([
    ('Input',                     f'(None, 1, {N_FEATURES})'),
    ('ZeroPad1D(1)',               f'(None, 3, {N_FEATURES})'),
    ('Conv1D 64, k=3 + ReLU + Dropout', '(None, 1, 64)'),
    ('ZeroPad1D(1)',               '(None, 3, 64)'),
    ('Conv1D 32, k=3 + ReLU',     '(None, 1, 32)'),
    ('Flatten',                   '(None, 32)'),
    ('Dense 32 + ReLU + Dropout', '(None, 32)'),
    ('Dense  1 (Output)',         '(None,  1)'),
], title='CNN-1D Architecture', color=COLOR_MAP['CNN-1D'],
   filename='fig06b_cnn_arch.png')

In [ ]:
cnn_history = fit_dl(cnn_model, 'CNN', X_train_seq, y_train_s, X_val_seq, y_val_s)
cnn_preds, cnn_trues, cnn_metrics = evaluate_dl(cnn_model, X_test_seq, y_test_s, 'CNN')

---
## 7. DL Model 3 — LSTM

In [ ]:
def build_lstm(n_features, hidden=64, dropout=0.2):
    inp = Input(shape=(1, n_features), name='input')
    x = layers.LSTM(hidden, dropout=dropout)(inp)
    x = layers.Dense(32, activation='relu')(x)
    x = layers.Dropout(dropout)(x)
    out = layers.Dense(1)(x)
    return Model(inp, out, name='LSTM')

lstm_model = build_lstm(N_FEATURES)
lstm_model.summary()

In [ ]:
draw_architecture([
    ('Input',                      f'(None, 1, {N_FEATURES})'),
    ('LSTM 64 + Dropout',          '(None, 64)'),
    ('Dense 32 + ReLU + Dropout',  '(None, 32)'),
    ('Dense  1 (Output)',          '(None,  1)'),
], title='LSTM Architecture', color=COLOR_MAP['LSTM'],
   filename='fig06c_lstm_arch.png')

In [ ]:
lstm_history = fit_dl(lstm_model, 'LSTM', X_train_seq, y_train_s, X_val_seq, y_val_s)
lstm_preds, lstm_trues, lstm_metrics = evaluate_dl(lstm_model, X_test_seq, y_test_s, 'LSTM')

---
## 8. DL Model 4 — Transformer

In [ ]:
def build_transformer(n_features, num_heads=2, key_dim=64, dropout=0.1):
    inp = Input(shape=(1, n_features), name='input')
    attn = layers.MultiHeadAttention(
        num_heads=num_heads, key_dim=key_dim, dropout=dropout)(inp, inp)
    attn = layers.LayerNormalization()(attn + inp)
    x = layers.GlobalAveragePooling1D()(attn)
    x = layers.Dense(64, activation='relu')(x)
    x = layers.Dropout(dropout)(x)
    x = layers.Dense(32, activation='relu')(x)
    out = layers.Dense(1)(x)
    return Model(inp, out, name='Transformer')

tfm_model = build_transformer(N_FEATURES)
tfm_model.summary()

In [ ]:
draw_architecture([
    ('Input',                       f'(None, 1, {N_FEATURES})'),
    ('MultiHeadAttention (2 heads)', '(None, 1, N_FEATURES)'),
    ('Add & LayerNorm',              '(None, 1, N_FEATURES)'),
    ('GlobalAveragePooling1D',       '(None, N_FEATURES)'),
    ('Dense 64 + ReLU + Dropout',    '(None, 64)'),
    ('Dense 32 + ReLU',              '(None, 32)'),
    ('Dense  1 (Output)',            '(None,  1)'),
], title='Transformer Architecture', color=COLOR_MAP['Transformer'],
   filename='fig06d_tfm_arch.png')

In [ ]:
tfm_history = fit_dl(tfm_model, 'Transformer', X_train_seq, y_train_s, X_val_seq, y_val_s)
tfm_preds, tfm_trues, tfm_metrics = evaluate_dl(tfm_model, X_test_seq, y_test_s, 'Transformer')

---
## 9. Shallow ML Models

All shallow models receive **scaled** features (`X_train_s`) but **original-scale** targets  
(no `y_scaler` needed — sklearn regressors are not sensitive to target scale in the same  
way gradient-descent networks are).

### 9.1 Random Forest

In [ ]:
rf_model = RandomForestRegressor(
    n_estimators=300,
    max_depth=None,
    min_samples_leaf=2,
    n_jobs=-1,
    random_state=SEED
)
rf_model.fit(X_train_s, y_train_flat)
rf_preds, rf_trues, rf_metrics = evaluate_ml(rf_model, X_test_s, y_test_flat, 'Random Forest')

### 9.2 XGBoost

In [ ]:
if XGB_AVAILABLE:
    xgb_model = xgb.XGBRegressor(
        n_estimators=500,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=0.1,
        reg_lambda=1.0,
        random_state=SEED,
        n_jobs=-1,
        verbosity=0
    )
    xgb_model.fit(
        X_train_s, y_train_flat,
        eval_set=[(X_val_s, y_val_flat)],
        verbose=False
    )
    xgb_preds, xgb_trues, xgb_metrics = evaluate_ml(xgb_model, X_test_s, y_test_flat, 'XGBoost')
else:
    print('Skipping XGBoost — not installed.')
    xgb_model, xgb_preds, xgb_trues = None, None, None
    xgb_metrics = {'RMSE': np.nan, 'MAE': np.nan, 'R2': np.nan, 'MAPE': np.nan}

### 9.3 Support Vector Regressor (SVR)

In [ ]:
# SVR is scale-sensitive and already receives standardised features.
# RBF kernel with C=10, epsilon=0.01 works well on engineering regression tasks.
svr_model = SVR(kernel='rbf', C=10, epsilon=0.01, gamma='scale')
svr_model.fit(X_train_s, y_train_flat)
svr_preds, svr_trues, svr_metrics = evaluate_ml(svr_model, X_test_s, y_test_flat, 'SVR')

### 9.4 Gradient Boosting

In [ ]:
gbm_model = GradientBoostingRegressor(
    n_estimators=400,
    learning_rate=0.05,
    max_depth=5,
    subsample=0.8,
    min_samples_leaf=3,
    random_state=SEED
)
gbm_model.fit(X_train_s, y_train_flat)
gbm_preds, gbm_trues, gbm_metrics = evaluate_ml(gbm_model, X_test_s, y_test_flat, 'Gradient Boosting')

---
## 10. Combined Results

In [ ]:
# Collect all predictions in one dict for plotting
all_preds = {
    'MLP':               (mlp_trues,  mlp_preds),
    'CNN-1D':            (cnn_trues,  cnn_preds),
    'LSTM':              (lstm_trues, lstm_preds),
    'Transformer':       (tfm_trues,  tfm_preds),
    'Random Forest':     (rf_trues,   rf_preds),
    'XGBoost':           (xgb_trues,  xgb_preds),
    'SVR':               (svr_trues,  svr_preds),
    'Gradient Boosting': (gbm_trues,  gbm_preds),
}

all_metrics = {
    'MLP':               mlp_metrics,
    'CNN-1D':            cnn_metrics,
    'LSTM':              lstm_metrics,
    'Transformer':       tfm_metrics,
    'Random Forest':     rf_metrics,
    'XGBoost':           xgb_metrics,
    'SVR':               svr_metrics,
    'Gradient Boosting': gbm_metrics,
}

results = pd.DataFrame([
    {'Model': name,
     'Type':  'Deep Learning' if name in DL_MODELS else 'Shallow ML',
     'RMSE':  m['RMSE'], 'MAE': m['MAE'],
     'R²':    m['R2'],   'MAPE (%)': m['MAPE']}
    for name, m in all_metrics.items()
]).sort_values('RMSE').reset_index(drop=True)

print('\n' + '='*75)
print('       FINAL COMPARISON — DL vs SHALLOW ML (Oil Recovery Prediction)')
print('='*75)
print(results.to_string(index=False))
print(f'\nBest model by RMSE : {results.iloc[0]["Model"]} ({results.iloc[0]["Type"]})')
print(f'Best model by R²   : {results.loc[results["R²"].idxmax(), "Model"]}')

results.style \
    .background_gradient(subset=['RMSE','MAE','MAPE (%)'], cmap='RdYlGn_r') \
    .background_gradient(subset=['R²'], cmap='RdYlGn') \
    .format({'RMSE':'{:.4f}','MAE':'{:.4f}','R²':'{:.4f}','MAPE (%)':'{:.2f}'})

---
## 11. Figures

### 11.1 DL Learning Curves (Fig 7)

In [ ]:
dl_histories = {'MLP': mlp_history, 'CNN-1D': cnn_history,
                'LSTM': lstm_history, 'Transformer': tfm_history}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for name, hist in dl_histories.items():
    c = COLOR_MAP[name]
    axes[0].plot(hist.history['loss'],     label=name, color=c, lw=1.8)
    axes[1].plot(hist.history['val_loss'], label=name, color=c, lw=1.8)
for ax, title in zip(axes, ['Training Loss (MSE)', 'Validation Loss (MSE)']):
    ax.set_title(title, fontsize=12)
    ax.set_xlabel('Epoch')
    ax.set_ylabel('MSE (scaled)')
    ax.legend()
    ax.grid(alpha=0.3)
plt.suptitle('DL Learning Curves', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('fig07_learning_curves.png', dpi=150, bbox_inches='tight')
plt.show()

### 11.2 Metric Bars — All 8 Models (Fig 8)

In [ ]:
model_order = results['Model'].tolist()
bar_colors  = [COLOR_MAP[m] for m in model_order]

fig, axes = plt.subplots(1, 4, figsize=(22, 5))
for ax, (metric, lower) in zip(
        axes, [('RMSE', True), ('MAE', True), ('R²', False), ('MAPE (%)', True)]):
    vals = results[metric].tolist()
    bars = ax.bar(model_order, vals, color=bar_colors, edgecolor='white', linewidth=1.1)
    ax.set_title(metric, fontsize=13, fontweight='bold')
    ax.set_xticklabels(model_order, rotation=30, ha='right', fontsize=8)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2,
                bar.get_height() + max(vals)*0.01,
                f'{v:.4f}', ha='center', va='bottom', fontsize=7)
    ax.set_xlabel('↓ better' if lower else '↑ better', fontsize=9, color='gray')
    ax.grid(axis='y', alpha=0.3)
plt.suptitle('Test-Set Metric Comparison — DL vs Shallow ML',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('fig08_metric_bars.png', dpi=150, bbox_inches='tight')
plt.show()

### 11.3 Actual vs Predicted — All 8 Models (Fig 9)

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(22, 10))
for ax, name in zip(axes.ravel(), model_order):
    t, p = all_preds[name]
    r2   = r2_score(t, p)
    rmse = np.sqrt(mean_squared_error(t, p))
    ax.scatter(t, p, alpha=0.35, s=12, color=COLOR_MAP[name], edgecolors='none')
    lo, hi = min(t.min(), p.min()), max(t.max(), p.max())
    ax.plot([lo, hi], [lo, hi], 'k--', lw=1.5)
    ax.set_title(f'{name}\nR²={r2:.4f}  RMSE={rmse:.4f}', fontsize=9)
    ax.set_xlabel('Actual (%)', fontsize=8)
    ax.set_ylabel('Predicted (%)', fontsize=8)
    ax.tick_params(labelsize=7)
    ax.grid(alpha=0.25)
plt.suptitle('Actual vs. Predicted — Test Set', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('fig09_actual_vs_predicted.png', dpi=150, bbox_inches='tight')
plt.show()

### 11.4 Residual Distributions (Fig 10)

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(22, 8))
for ax, name in zip(axes.ravel(), model_order):
    t, p = all_preds[name]
    res = t - p
    ax.hist(res, bins=50, color=COLOR_MAP[name], edgecolor='white', alpha=0.85)
    ax.axvline(0, color='black', linestyle='--', linewidth=1.5)
    ax.set_title(f'{name}\nmean={res.mean():.4f}', fontsize=9)
    ax.set_xlabel('Residual', fontsize=8)
    ax.set_ylabel('Count', fontsize=8)
    ax.tick_params(labelsize=7)
    ax.grid(alpha=0.25)
plt.suptitle('Residual Distributions — Test Set', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('fig10_residuals.png', dpi=150, bbox_inches='tight')
plt.show()

### 11.5 Residual vs Predicted (Fig 11)

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(22, 8))
for ax, name in zip(axes.ravel(), model_order):
    t, p = all_preds[name]
    res = t - p
    ax.scatter(p, res, alpha=0.3, s=10, color=COLOR_MAP[name], edgecolors='none')
    ax.axhline(0, color='black', linestyle='--', linewidth=1.5)
    ax.set_title(name, fontsize=9)
    ax.set_xlabel('Predicted (%)', fontsize=8)
    ax.set_ylabel('Residual', fontsize=8)
    ax.tick_params(labelsize=7)
    ax.grid(alpha=0.25)
plt.suptitle('Residual vs. Predicted — Homoscedasticity Check',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('fig11_residual_vs_predicted.png', dpi=150, bbox_inches='tight')
plt.show()

### 11.6 Absolute Error Box Plots (Fig 12)

In [ ]:
abs_errors = [np.abs(all_preds[m][0] - all_preds[m][1]) for m in model_order]

fig, ax = plt.subplots(figsize=(13, 5))
bp = ax.boxplot(abs_errors, labels=model_order,
                patch_artist=True, notch=False,
                medianprops=dict(color='black', linewidth=2))
for patch, name in zip(bp['boxes'], model_order):
    patch.set_facecolor(COLOR_MAP[name])
    patch.set_alpha(0.75)
ax.set_title('Absolute Error Distribution — Test Set', fontsize=13, fontweight='bold')
ax.set_ylabel('|Actual − Predicted| (%)')
ax.set_xticklabels(model_order, rotation=20, ha='right')
ax.grid(axis='y', alpha=0.3)

# Shade DL vs ML regions
ax.axvspan(0.5, 4.5,  alpha=0.05, color='blue',  label='Deep Learning')
ax.axvspan(4.5, 8.5,  alpha=0.05, color='red',   label='Shallow ML')
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig('fig12_abs_error_boxplot.png', dpi=150, bbox_inches='tight')
plt.show()

### 11.7 Radar Chart — All 8 Models (Fig 13)

In [ ]:
metric_cols  = ['RMSE', 'MAE', 'R²', 'MAPE (%)']
radar_labels = ['1-RMSE\n(norm)', '1-MAE\n(norm)', 'R²\n(norm)', '1-MAPE\n(norm)']

vals = results[metric_cols].values.astype(float)
norm = np.zeros_like(vals)
for j, col in enumerate(metric_cols):
    lo, hi = np.nanmin(vals[:, j]), np.nanmax(vals[:, j])
    if col == 'R²':
        norm[:, j] = (vals[:, j] - lo) / (hi - lo + 1e-12)
    else:
        norm[:, j] = 1 - (vals[:, j] - lo) / (hi - lo + 1e-12)

N = len(metric_cols)
angles = np.linspace(0, 2*np.pi, N, endpoint=False).tolist()
angles += angles[:1]

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))
for i, name in enumerate(model_order):
    ri = results.index[results['Model'] == name][0]
    v = norm[ri].tolist() + norm[ri][:1].tolist()
    ls = '-' if name in DL_MODELS else '--'
    ax.plot(angles, v, lw=2, color=COLOR_MAP[name], linestyle=ls, label=name)
    ax.fill(angles, v, alpha=0.05, color=COLOR_MAP[name])
ax.set_xticks(angles[:-1])
ax.set_xticklabels(radar_labels, size=11)
ax.set_ylim(0, 1)
ax.set_title('Radar Chart — All Models (outer = better)',
             fontsize=12, fontweight='bold', pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.45, 1.2), fontsize=8)
plt.tight_layout()
plt.savefig('fig13_radar_chart.png', dpi=150, bbox_inches='tight')
plt.show()

### 11.8 DL Complexity vs Accuracy (Fig 14)

In [ ]:
dl_params = {
    'MLP':         mlp_model.count_params(),
    'CNN-1D':      cnn_model.count_params(),
    'LSTM':        lstm_model.count_params(),
    'Transformer': tfm_model.count_params(),
}

fig, ax = plt.subplots(figsize=(8, 5))
for name, params in dl_params.items():
    rmse = all_metrics[name]['RMSE']
    ax.scatter(params, rmse, s=200, color=COLOR_MAP[name], zorder=3, label=name)
    ax.annotate(name, (params, rmse),
                textcoords='offset points', xytext=(8, 4), fontsize=10)
ax.set_xlabel('Trainable Parameters (DL models)')
ax.set_ylabel('Test RMSE')
ax.set_title('DL Model Complexity vs. Accuracy', fontsize=13, fontweight='bold')
ax.legend(fontsize=9)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('fig14_complexity_vs_accuracy.png', dpi=150, bbox_inches='tight')
plt.show()

### 11.9 DL vs ML — RMSE & R² Side-by-Side (Fig 14b)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
x = np.arange(len(model_order))
width = 0.6

rmse_vals = [all_metrics[m]['RMSE'] for m in model_order]
r2_vals   = [all_metrics[m]['R2']   for m in model_order]

bars0 = axes[0].bar(x, rmse_vals, width, color=[COLOR_MAP[m] for m in model_order],
                    edgecolor='white')
axes[0].set_xticks(x)
axes[0].set_xticklabels(model_order, rotation=30, ha='right', fontsize=9)
axes[0].set_ylabel('RMSE  (↓ better)')
axes[0].set_title('RMSE Comparison', fontsize=12, fontweight='bold')
axes[0].axvline(3.5, color='gray', linestyle='--', lw=1, label='DL | ML boundary')
axes[0].legend(fontsize=8)
axes[0].grid(axis='y', alpha=0.3)
for bar, v in zip(bars0, rmse_vals):
    axes[0].text(bar.get_x() + bar.get_width()/2,
                 bar.get_height() + max(rmse_vals)*0.01,
                 f'{v:.4f}', ha='center', va='bottom', fontsize=7)

bars1 = axes[1].bar(x, r2_vals, width, color=[COLOR_MAP[m] for m in model_order],
                    edgecolor='white')
axes[1].set_xticks(x)
axes[1].set_xticklabels(model_order, rotation=30, ha='right', fontsize=9)
axes[1].set_ylabel('R²  (↑ better)')
axes[1].set_title('R² Comparison', fontsize=12, fontweight='bold')
axes[1].axvline(3.5, color='gray', linestyle='--', lw=1, label='DL | ML boundary')
axes[1].legend(fontsize=8)
axes[1].grid(axis='y', alpha=0.3)
for bar, v in zip(bars1, r2_vals):
    axes[1].text(bar.get_x() + bar.get_width()/2,
                 bar.get_height() + max(r2_vals)*0.005,
                 f'{v:.4f}', ha='center', va='bottom', fontsize=7)

plt.suptitle('Deep Learning vs Shallow ML — RMSE & R² on Test Set',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('fig14b_dl_vs_ml_bars.png', dpi=150, bbox_inches='tight')
plt.show()

### 11.10 Permutation Feature Importance — Best Overall Model (Fig 15)

In [ ]:
def permutation_importance_dl(model, X_s, y_inv, y_scaler,
                               feature_names, n_repeats=5, is_seq=False):
    X_input   = X_s.reshape(-1, 1, X_s.shape[1]) if is_seq else X_s
    base      = y_scaler.inverse_transform(model.predict(X_input, verbose=0))
    base_rmse = np.sqrt(mean_squared_error(y_inv, base))
    imp = np.zeros((len(feature_names), n_repeats))
    for i in range(len(feature_names)):
        for r in range(n_repeats):
            Xp = X_s.copy()
            np.random.shuffle(Xp[:, i])
            Xp_in = Xp.reshape(-1, 1, Xp.shape[1]) if is_seq else Xp
            pred  = y_scaler.inverse_transform(model.predict(Xp_in, verbose=0))
            imp[i, r] = np.sqrt(mean_squared_error(y_inv, pred)) - base_rmse
    return imp.mean(axis=1), imp.std(axis=1)


def permutation_importance_sklearn(model, X_s, y_true,
                                    feature_names, n_repeats=5):
    base_rmse = np.sqrt(mean_squared_error(y_true, model.predict(X_s)))
    imp = np.zeros((len(feature_names), n_repeats))
    for i in range(len(feature_names)):
        for r in range(n_repeats):
            Xp = X_s.copy()
            np.random.shuffle(Xp[:, i])
            imp[i, r] = np.sqrt(mean_squared_error(y_true, model.predict(Xp))) - base_rmse
    return imp.mean(axis=1), imp.std(axis=1)


best_name = results.iloc[0]['Model']
print(f'Best model: {best_name}')
y_test_inv = y_test_flat  # original scale

dl_map = {'MLP': (mlp_model, False), 'CNN-1D': (cnn_model, True),
           'LSTM': (lstm_model, True), 'Transformer': (tfm_model, True)}
ml_map = {'Random Forest': rf_model, 'XGBoost': xgb_model,
           'SVR': svr_model, 'Gradient Boosting': gbm_model}

if best_name in dl_map:
    bm, is_seq = dl_map[best_name]
    imp_mean, imp_std = permutation_importance_dl(
        bm, X_test_s, y_test_inv.reshape(-1,1), y_scaler, FEATURES, is_seq=is_seq)
else:
    bm = ml_map[best_name]
    imp_mean, imp_std = permutation_importance_sklearn(
        bm, X_test_s, y_test_inv, FEATURES)

sorted_idx = np.argsort(imp_mean)[::-1]

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

axes[0].barh([FEATURES[i] for i in sorted_idx],
             imp_mean[sorted_idx], xerr=imp_std[sorted_idx],
             color=COLOR_MAP[best_name], alpha=0.85, edgecolor='white')
axes[0].set_xlabel('Mean RMSE increase')
axes[0].set_title(f'Permutation Importance — {best_name}',
                  fontsize=12, fontweight='bold')
axes[0].invert_yaxis()
axes[0].grid(axis='x', alpha=0.3)

imp_df = pd.DataFrame({'Feature': FEATURES, 'Importance': imp_mean}).set_index('Feature')
imp_sorted = imp_df.sort_values('Importance', ascending=False)
sns.heatmap(imp_sorted[['Importance']].T, ax=axes[1],
            cmap='YlOrRd', annot=True, fmt='.4f', linewidths=0.5,
            cbar_kws={'label': 'RMSE increase'}, annot_kws={'size': 7})
axes[1].set_title('Feature Importance Heatmap', fontsize=12, fontweight='bold')
axes[1].set_yticklabels(['Importance'], rotation=0)
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=45, ha='right', fontsize=8)

plt.suptitle(f'Feature Importance — {best_name} (Best Model)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('fig15_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

### 11.11 Prediction Error Band — All 8 Models (Fig 16)

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(22, 10))
for ax, name in zip(axes.ravel(), model_order):
    t, p = all_preds[name]
    idx  = np.argsort(t)
    tv, pv = t[idx], p[idx]
    x = np.arange(len(tv))
    ax.fill_between(x, tv, pv, alpha=0.35, color=COLOR_MAP[name], label='Error band')
    ax.plot(x, tv, 'k-',  lw=0.8, label='Actual')
    ax.plot(x, pv, '--',  lw=0.8, color=COLOR_MAP[name], label='Predicted')
    ax.set_title(f'{name}  MAE={np.abs(tv-pv).mean():.4f}', fontsize=9)
    ax.set_xlabel('Sorted test index', fontsize=8)
    ax.set_ylabel('Oil Recovery (%)', fontsize=8)
    ax.tick_params(labelsize=7)
    ax.legend(fontsize=7)
    ax.grid(alpha=0.25)
plt.suptitle('Prediction Error Band (sorted by actual value)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('fig16_error_band.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 12. Summary

The table below ranks all 8 models by test-set RMSE and highlights which family wins.

In [ ]:
summary = results[['Model','Type','RMSE','MAE','R²','MAPE (%)']].copy()
summary['Rank'] = range(1, len(summary)+1)

print('\n' + '='*80)
print('            FINAL RANKING — Deep Learning vs Shallow ML')
print('='*80)
print(summary[['Rank','Model','Type','RMSE','MAE','R²','MAPE (%)']].to_string(index=False))
print()

dl_avg_r2 = results.loc[results['Type']=='Deep Learning', 'R²'].mean()
ml_avg_r2 = results.loc[results['Type']=='Shallow ML',    'R²'].mean()
print(f'Average R²  — Deep Learning : {dl_avg_r2:.4f}')
print(f'Average R²  — Shallow ML    : {ml_avg_r2:.4f}')
winner = 'Deep Learning' if dl_avg_r2 > ml_avg_r2 else 'Shallow ML'
print(f'\n>>> Higher average R²: {winner} <<<')